## 1. Найти корень числа (ближайшее целое)

In [4]:
def mySqrt(x: int) -> int:
    """
    Находит целую часть квадратного корня числа x (округление вниз) с помощью бинарного поиска.
    
    ОПРЕДЕЛЕНИЕ ГРАНИЦ ПОИСКА:
    - Левая граница (l = 0): Корень неотрицательного числа не может быть меньше 0.
    - Правая граница (r = x): Для любого целого x >= 0 квадратный корень не превосходит x (sqrt(x) <= x).
    Искомое значение лежит на отсортированном отрезке [0, x].
    
    СЛОЖНОСТЬ O(log x):
    На каждом шаге цикла мы берем середину отрезка m и отбрасываем ровно половину 
    поискового пространства. Длина интервала (x + 1) уменьшается в 2 раза на каждой 
    итерации, поэтому максимальное число шагов равно log2(x), что дает сложность O(log x).
    Сложность по памяти — O(1).
    """
    # 1. Границы поиска: значение sqrt(x) гарантированно лежит в диапазоне [0, x]
    l = 0
    r = x
    res = 0
    
    while l <= r:
        # Берём середину текущего интервала (l + (r - l) // 2 предотвращает переполнение)
        m = l + ((r - l) // 2)
        
        # Если m^2 больше x, значит искомый корень левее -> сужаем правую границу r
        if m ** 2 > x:
            r = m - 1
            
        # Если m^2 меньше x, то m — потенциальный кандидат на ответ (округление вниз)
        # Сохраняем m в res и пытаемся найти большее подходящее число справа
        elif m ** 2 < x:
            l = m + 1
            res = m
            
        # Если m^2 == x, мы нашли точный корень -> мгновенный возврат
        else:
            return m
            
    return res

## 2. Очень лёгкая задача (два ксерокса) 

In [5]:
def min_copy_time(N: int, x: int, y: int) -> int:
    """
    Находит минимальное время для изготовления N копий на двух ксероксах со скоростями x и y.
    
    ОПРЕДЕЛЕНИЕ ГРАНИЦ ПОИСКА ([l, r]):
    - Левая граница (l = 0): Минимально возможное время равно 0 (при N = 0).
    - Правая граница (r): Максимальное время ограничено худшим ксероксом, умноженным на N: 
      max(x, y) * N. Даже если бы работал только один медленный ксерокс, он бы справился за это время.
    
    СЛОЖНОСТЬ O(log(max(x, y) * N)):
    Бинарный поиск делит интервал [0, max(x, y) * N] пополам на каждой итерации, 
    что дает логарифмическую сложность.
    """
    # Если нужно сделать 0 копий, времени не требуется
    if N == 0:
        return 0

    # Ускорим процесс: пусть x будет быстрее (меньше времени на 1 копию), а y — медленнее
    if x > y:
        x, y = y, x

    # Если нужно сделать всего 1 копию, ее делает самый быстрый ксерокс
    if N == 1:
        return x

    l = 0
    r = x * N  # Максимальное время ограничено работой только быстрого ксерокса для всех N копий
    best_time = r

    while l <= r:
        # Середина временного интервала, которую мы проверяем
        mid = l + (r - l) // 2

        # ПРОВЕРКА УСЛОВИЯ ДОСТАТОЧНОСТИ ВРЕМЕНИ (mid):
        # 1. Сначала на первую копию тратится время самого быстрого ксерокса (x).
        #    Без этого один ксерокс начал бы работу на секунду раньше другого, 
        #    поэтому мы как бы «выделяем» x секунд на старт, после чего оба аппарата 
        #    работают параллельно.
        # 2. Оставшееся время (mid - x) делим на скорость каждого ксерокса и суммируем копии.
        # 3. Если (минус защита от отрицательного времени, если mid < x) общее число 
        #    сделанных копий >= N, то времени (mid) достаточно.
        if mid < x:
            copies_made = 0
        else:
            copies_made = 1 + (mid - x) // x + (mid - x) // y

        # Если за время `mid` ксероксы успевают сделать N или более копий:
        if copies_made >= N:
            best_time = mid  # Запоминаем успешное время как потенциальный ответ
            r = mid - 1      # Пытаемся найти еще меньшее (более оптимальное) время в левой половине
        else:
            # Если копий не хватает, значит, времени слишком мало -> сдвигаем левую границу вправо
            l = mid + 1

    return best_time


## 3. Накормить животных 

In [3]:
from typing import List

def max_fed_animals(animals: List[int], portions: List[int]) -> int:
    """
    Определяет максимальное количество животных, которых можно накормить.
    
    ПОЧЕМУ ЭТА ЖАДНАЯ СТРАТЕГИЯ ДАЕТ МАКСИМУМ (Обоснование):
    1. Сортировка: Мы сортируем аппетиты животных и размеры порций по возрастанию.
    2. Жадный выбор: Мы всегда стараемся накормить животное с минимальным требуемым 
       количеством еды самой маленькой из доступных порций, которая его удовлетворяет.
    3. Доказательство оптимальности: 
       - Если мы используем большую порцию для животного с маленьким аппетитом, 
         мы «тратим впустую» ресурс, который мог бы понадобиться более требовательному 
         животному, в то время как маленькая порция осталась бы невостребованной.
       - Если мы попытаемся дать животному с большим аппетитом маленькую порцию, 
         мы вообще не сможем его накормить, а порция сгорит впустую.
       Сортировка и жадное сопоставление «от меньшего к большему» гарантируют, 
       что мы максимально эффективно расходуем каждую порцию и не уступаем в количестве 
       удовлетворенных животных ни одному другому варианту распределения.
    
    Сложность алгоритма: O(N log N + M log M) из-за сортировки, где N и M — 
    количество животных и порций соответственно. Пространственная сложность: O(1) или O(N) 
    в зависимости от реализации сортировки.
    """
    # Сортируем списки требований животных и размеров порций по возрастанию
    animals.sort()
    portions.sort()
    
    animal_ptr = 0  # Указатель на текущее животное
    portion_ptr = 0 # Указатель на текущую порцию еды
    
    # Проходим по обоим спискам с помощью двух указателей
    while animal_ptr < len(animals) and portion_ptr < len(portions):
        # Если текущая порция еды больше или равна требуемой для животного
        if portions[portion_ptr] >= animals[animal_ptr]:
            # Животное успешно накормлено, переходим к следующему животному
            animal_ptr += 1
        
        # В любом случае переходим к рассмотрению следующей порции. 
        # (Если порция подошла, мы её отдали. Если была слишком мала — она никому 
        # из оставшихся еще более требовательных животных тем более не подойдет).
        portion_ptr += 1
        
    # Итоговое значение animal_ptr показывает, сколько животных удалось накормить
    return animal_ptr



## 4. Найти разницу между двумя строками

In [7]:
def findTheDifference(a: str, b: str) -> str:
    """
    Находит добавленный символ в строке b по сравнению со строкой a.
    
    ОБЪЯСНЕНИЕ РАБОТЫ АЛГОРИТМА (ЧЕРЕЗ XOR):
    Операция исключающее ИЛИ (XOR, в Python обозначается как ^) обладает следующими свойствами:
    1. x ^ x = 0 (XOR любого числа с самим собой дает ноль).
    2. x ^ 0 = x (XOR числа с нулем не меняет его).
    3. Операция коммутативна и ассоциативна (порядок вычислений не имеет значения).
    
    Как это находит лишний элемент?
    - Если мы применим XOR ко всем символам обеих строк (a и b), то все одинаковые 
      символы встретятся чётное число раз (один раз в a и один раз в b).
    - Из-за свойства x ^ x = 0 все парные символы взаимно уничтожатся («сократятся»).
    - Единственный добавленный символ из строки b останется без пары (встретится нечётное 
      число раз), и его код в итоге останется в переменной res.
    
    Альтернативный подход через подсчёт символов (через collections.Counter):
    Можно было бы поспать частоты символов в обеих строках и вычесть их (Counter(b) - Counter(a)), 
    что также работает за O(n), но требует O( мощности алфавита ) дополнительной памяти.
    XOR же элегантнее тем, что требует O(1) памяти.
    """
    res = 0
    
    # Побитово складываем (XOR-им) все символы первой строки
    for char in a:
        res ^= ord(char)
        
    # Продолжаем XOR-ить символы второй строки (включая лишний)
    for char in b:
        res ^= ord(char)
        
    # В res остался ASCII-код (или юникод) добавленного символа. 
    # Преобразуем его обратно в букву с помощью функции chr().
    return chr(res)


## 5. Сумма двух элементов массива

In [13]:
from typing import List

def twoSum(nums: List[int], target: int) -> List[int]:
    """
    Находит индексы двух элементов массива, сумма которых равна target, за O(n).
    
    ОБЪЯСНЕНИЕ: КАК ХРАНЕНИЕ ПРОСМОТРЕННЫХ ЭЛЕМЕНТОВ ПОЗВОЛЯЕТ ИЗБЕЖАТЬ ДВОЙНОГО ЦИКЛА:
    1. Наивный подход с двойным циклом (O(n^2)) для каждого элемента заново 
       сканировал бы весь остаток массива в поисках подходящей пары.
    2. Вместо этого мы используем хеш-таблицу (словарь `seen`), в которой по ходу 
       прохода сохраняем все уже встреченные числа и их индексы: {значение: индекс}.
    3. Для каждого текущего элемента `num` мы мгновенно (за O(1)) вычисляем его 
       «дополнение» (complement): complement = target - num.
    4. Если это дополнение *уже есть* в нашем словаре просмотренных элементов, 
       значит, мы нашли искомую пару прямо на лету, не перебирая массив повторно!
    5. Если дополнения пока нет, мы просто добавляем текущий элемент в словарь 
       и идем дальше. Итог: один проход по массиву и сложность O(n).
    """
    history = {}  # Словарь для хранения уже просмотренных элементов: { число: индекс }
    
    for i, num in enumerate(nums):
        A = target - num  # Искомое число, которого не хватает до target
        
        # Проверяем, встречали ли мы уже нужное дополнение ранее
        if A in history:
            return [history[A], i]  # Возвращаем индексы найденной пары
        
        # Если нет — сохраняем текущее число в словарь и продолжаем поиск
        history[num] = i
        
    return []  # Возвращаем пустой массив, если подходящей пары не нашлось


## 6. Массив анаграмм 

In [14]:
from typing import List

def groupAnagrams(strs: list[str]) -> list[list[str]]:
    # Словарь для группировки: ключ — кортеж частот символов, значение — список анаграмм
    ans = {}

    for s in strs:
        # Создаем массив из 26 нулей для подсчета каждой буквы английского алфавита (от 'a' до 'z')
        count = [0] * 26
        for char in s:
            # Вычисляем индекс буквы в массиве и увеличиваем счетчик на 1
            count[ord(char) - ord("a")] += 1

        # Преобразуем list в tuple, так как list изменяемый и не может быть ключом словаря (unhashable),
        # а tuple неизменяемый и хэшируем!
        key = tuple(count)

        # Если этой комбинации символов еще нет в словаре, инициализируем для нее пустой список
        if key not in ans:
            ans[key] = []
            
        # Добавляем исходную строку в соответствующую группу анаграмм
        ans[key].append(s)

    # Возвращаем только значения словаря в виде списка списков
    return list(ans.values())